# Train / Validation / Test Splits with Time-Series Features

We partition the historical portion of `data/processed/merged_data_model_ready_interactions.csv` (2024-03-14 → 2025-08-17) into train/validation/test folds. Rows from 2025-08-18 → 2025-09-14 remain untouched as the final forecast horizon.

**Default cutoffs (adjust as needed):**
- Train: 2024-03-14 → 2025-05-31
- Validation: 2025-06-01 → 2025-06-30
- Test: 2025-07-01 → 2025-08-17
- Holdout: 2025-08-18 → 2025-09-14 (not used for fitting/eval; reserved for final scoring)

## Imports & Parameters

In [8]:

import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'data' / 'processed' / 'merged_data_model_ready_interactions.csv'
OUTPUT_DIR = BASE_DIR / 'data' / 'processed' / 'timeseries_splits'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Single aligned split
TRAIN_END = pd.Timestamp('2025-06-30')
VAL_START = pd.Timestamp('2025-07-01')
VAL_END = pd.Timestamp('2025-07-28')
TEST_START = pd.Timestamp('2025-07-29')
TEST_END = pd.Timestamp('2025-08-17')
HOLDOUT_START = pd.Timestamp('2025-08-18')

print('Train end:', TRAIN_END.date())
print('Validation:', VAL_START.date(), '→', VAL_END.date())
print('Test:', TEST_START.date(), '→', TEST_END.date())
print('Holdout starts:', HOLDOUT_START.date())


Validation folds:
  fold1: 2025-05-03 → 2025-05-30
  fold2: 2025-05-17 → 2025-06-13
  fold3: 2025-05-31 → 2025-06-27
Test window: 2025-07-01 → 2025-08-17
Holdout starts: 2025-08-18


## Load Final Dataset

In [9]:

df = pd.read_csv(DATA_PATH, parse_dates=['dt'], low_memory=False)
print('Shape:', df.shape)
print('Date range:', df['dt'].min().date(), '→', df['dt'].max().date())
print('Departments:', df['dept_id'].unique().tolist())
df.head()


Shape: (275000, 72)
Date range: 2024-03-14 → 2025-09-14
Departments: [6, 90, 9, 41, 67]


,dept_id,store_id,dt,cases,trucks,state_name,market_area_nbr,region_nbr,dept_desc,gmm_name,...,trend_party_interaction,trend_dairy_interaction,trend_cameras_interaction,bts_sporting_flag,bts_celebration_flag,sports_event_sporting_flag,hot_back_to_school_flag,sales_tax_back_to_school_flag,sales_tax_bts_sporting,cpi_food_gap
0,6,10002,2025-02-20,63.0,2.0,NC,296,26,CAMERAS AND SUPPLIES,GENERAL MERCHANDISE,...,0.0,0.00,0.64,0,0,0,0,0,0,13.675
1,6,10002,2025-02-28,56.0,2.0,NC,296,26,CAMERAS AND SUPPLIES,GENERAL MERCHANDISE,...,0.0,0.00,0.44,0,0,0,0,0,0,13.675
2,6,10002,2025-02-19,62.0,2.0,NC,296,26,CAMERAS AND SUPPLIES,GENERAL MERCHANDISE,...,0.0,0.00,0.64,0,0,0,0,0,0,13.675
3,90,10001,2025-02-25,67.0,3.0,MD,285,22,DAIRY,FOOD,...,0.0,0.02,0.00,0,0,0,0,0,0,13.675
4,9,10004,2025-02-07,53.0,2.0,LA,66,13,SPORTING GOODS,GENERAL MERCHANDISE,...,0.0,0.00,0.00,0,0,0,0,0,0,13.675


## Split by Date

In [10]:

df = pd.read_csv(DATA_PATH, parse_dates=['dt'], low_memory=False)
print('Shape:', df.shape)
print('Date range:', df['dt'].min().date(), '→', df['dt'].max().date())
print('Departments:', df['dept_id'].unique().tolist())

train_mask = df['dt'] <= TRAIN_END
val_mask = (df['dt'] >= VAL_START) & (df['dt'] <= VAL_END)
test_mask = (df['dt'] >= TEST_START) & (df['dt'] <= TEST_END)
holdout_mask = df['dt'] >= HOLDOUT_START

train_df = df.loc[train_mask].copy()
val_df = df.loc[val_mask].copy()
test_df = df.loc[test_mask].copy()
holdout_df = df.loc[holdout_mask].copy()

print('Train rows:', len(train_df))
print('Validation rows:', len(val_df))
print('Test rows:', len(test_df))
print('Holdout rows:', len(holdout_df))


History rows (train+val+test): 261000
Holdout rows: 14000



## Integrity check
Ensure no duplicate store–dept–date keys and confirm history/holdout boundaries align with cutoffs.


In [11]:

# Sanity checks
key_dupes = df.duplicated(['store_id','dept_id','dt']).sum()
if key_dupes:
    raise ValueError(f'Duplicate store/dept/dt rows found: {key_dupes}')

train_max = train_df['dt'].max()
val_min = val_df['dt'].min() if len(val_df) else None
val_max = val_df['dt'].max() if len(val_df) else None
test_min = test_df['dt'].min() if len(test_df) else None
holdout_min = holdout_df['dt'].min() if len(holdout_df) else None
print('Sanity checks passed:',
      '| train max =', train_max.date(),
      '| val window =', val_min.date() if val_min is not None else 'N/A', '→', val_max.date() if val_max is not None else 'N/A',
      '| test window =', test_min.date() if test_min is not None else 'N/A', '→', TEST_END.date(),
      '| holdout min =', holdout_min.date() if holdout_min is not None else 'N/A')


Sanity checks passed: no duplicate keys; history max = 2025-08-17 | holdout min = 2025-08-18



## Helper: Build Lag/Rolling Features Safely

We define a function that, given a DataFrame (train/val/test), computes lagged and rolling statistics per `(store_id, dept_id)` using only past data. For validation/test, we prepend the previous splits' history so the rolling window has context, then drop the extra rows.


In [12]:


def add_lag_features(target_df, history_df=None, group_cols=("store_id","dept_id"), date_col='dt'):
    """Return target_df with lag/rolling features computed only from prior dates.

    history_df provides preceding rows (e.g., train or train+val) so validation/test/holdout
    features never leak future information.
    """
    if history_df is None:
        history_df = target_df.head(0)

    concat_df = pd.concat([history_df, target_df], axis=0)
    concat_df = concat_df.sort_values(list(group_cols) + [date_col])

    # Forward-fill targets so forecast rows inherit last observed values instead of propagating NaNs
    concat_df['cases_ffill'] = concat_df.groupby(list(group_cols))['cases'].ffill()
    concat_df['trucks_ffill'] = concat_df.groupby(list(group_cols))['trucks'].ffill()

    grouped_cases = concat_df.groupby(list(group_cols))['cases_ffill']
    grouped_trucks = concat_df.groupby(list(group_cols))['trucks_ffill']

    for lag in [1, 7, 14]:
        concat_df[f'cases_lag_{lag}'] = grouped_cases.shift(lag)

    concat_df['cases_ma_7'] = grouped_cases.shift(1).rolling(window=7, min_periods=1).mean()
    concat_df['cases_ma_14'] = grouped_cases.shift(1).rolling(window=14, min_periods=1).mean()
    concat_df['cases_std_7'] = grouped_cases.shift(1).rolling(window=7, min_periods=1).std()

    concat_df['trucks_lag_1'] = grouped_trucks.shift(1)
    concat_df['trucks_lag_7'] = grouped_trucks.shift(7)
    concat_df['trucks_ma_7'] = grouped_trucks.shift(1).rolling(window=7, min_periods=1).mean()

    out = concat_df.loc[target_df.index].copy()
    out.drop(columns=['cases_ffill','trucks_ffill'], inplace=True, errors='ignore')
    return out


Helper function builds leak-free lag/rolling features (1/7/14 day lags; 7/14 day rolling means, std; trucks 1/7 lags + 7-day mean) and uses optional history_df so validation/test/holdout windows never peek forward.

In [13]:

# Add lag features per split
train_with_lags = add_lag_features(train_df)
val_with_lags = add_lag_features(val_df, history_df=train_df)
train_val_history = pd.concat([train_df, val_df], axis=0)
test_with_lags = add_lag_features(test_df, history_df=train_val_history)

history_through_test = pd.concat([train_val_history, test_df], axis=0)
holdout_with_lags = add_lag_features(holdout_df, history_df=history_through_test)

for name, frame in {
    'Train': train_with_lags,
    'Validation': val_with_lags,
    'Test': test_with_lags,
    'Holdout': holdout_with_lags,
}.items():
    lag_nulls = frame[['cases_lag_1','cases_lag_7','cases_lag_14']].isnull().sum().to_dict()
    print(f"{name} lag nulls:", lag_nulls)


fold1 validation lag nulls: {'cases_lag_1': 0, 'cases_lag_7': 0, 'cases_lag_14': 0}
fold2 validation lag nulls: {'cases_lag_1': 0, 'cases_lag_7': 0, 'cases_lag_14': 0}
fold3 validation lag nulls: {'cases_lag_1': 0, 'cases_lag_7': 0, 'cases_lag_14': 0}
Test lag nulls: {'cases_lag_1': 0, 'cases_lag_7': 0, 'cases_lag_14': 0}
Holdout lag nulls: {'cases_lag_1': 0, 'cases_lag_7': 0, 'cases_lag_14': 0}


## Save Splits

In [14]:

# Cleanup old outputs
for fname in ['fold1_train_timeseries.csv','fold1_val_timeseries.csv','fold2_train_timeseries.csv','fold2_val_timeseries.csv','fold3_train_timeseries.csv','fold3_val_timeseries.csv']:
    try:
        (OUTPUT_DIR / fname).unlink()
    except FileNotFoundError:
        pass

train_with_lags.to_csv(OUTPUT_DIR / 'train_timeseries.csv', index=False)
val_with_lags.to_csv(OUTPUT_DIR / 'val_timeseries.csv', index=False)
test_with_lags.to_csv(OUTPUT_DIR / 'test_timeseries.csv', index=False)
holdout_with_lags.to_csv(OUTPUT_DIR / 'holdout_forecast_window.csv', index=False)

print('Saved train/val/test/holdout files to', OUTPUT_DIR)


Saved rolling-origin folds, test, and holdout to /Users/chanamalluvinay/Documents/wmt_proj/data/processed/timeseries_splits



## Notes
- Adjust `TRAIN_END` and `VAL_END` at the top if you need different cutoffs.
- The lag features currently cover cases/trucks; extend the helper function for additional metrics as needed.
- Forecast rows (with `cases` null) live beyond the test split and can be scored after a model is trained.
- External signals in the forecast window are frozen to the last historical values (per the prep pipeline) to avoid future-data leakage; splits reflect that.
